# 준비 단계
추론 단계에서는 평가 모델에게 질문을 던져 추론시켰다면, 평가 단계에서는 그 결과물을 가지고 OpenAI의 GPT를 이용해서 리뷰한다. 이 때문에 OpenAI 패키지를 다운받고 미리 발급받아놓은 API 키를 프로젝트 내에 저장해둬야 한다. OpenAI의 gpt에 메시지를 보내고 응답만 받기 때문에 코랩 리소스는 CPU만으로도 충분하다.

https://github.com/prometheus-eval/prometheus-eval/blob/main/train/README.md

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cd /content/drive/MyDrive/OpenFedFLASK/FLASK/gpt_review

/content/drive/MyDrive/OpenFedFLASK/FLASK/gpt_review


In [ ]:
!pip install shortuuid

In [ ]:
!pip install openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 1.8 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.54.4
    Uninstalling openai-1.54.4:
      Successfully uninstalled openai-1.54.4


In [ ]:
!pip install jsonlines

In [ ]:
import jsonlines
import os
def create_empty_jsonl_file(file_path):
    # 빈 데이터
    data = []

    # 해당 경로에 파일이 존재하는지 확인
    if os.path.exists(file_path):
        print(f"Error: file '{file_path}' exists.")
    else:
        # 해당 경로에 빈 데이터를 가진 파일 생성
        with jsonlines.open(file_path, mode='w') as writer:
            writer.write_all(data)
        print(f"Empty JSONL file created at: {file_path}")

def create_empty_directory(directory_path):
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)
        print(f"Empty Directory created at: {directory_path}")
    else:
        print(f"Error: file '{directory_path}' exists.")

# 평가 진행
여기선 1700개 대신 10개짜리 추론 json 파일을 평가해볼텐데, 추론 json 파일과 평가 json 파일의 리스트 길이가 맞아떨어지지 않으면 오류가 발생하므로 주의한다.

gpt4-eval.py에 여러 부분에서 오류가 발생한다는 것이 가장 골치 아픈 문제이다. 주로 'question_id'나 'idx' 같은 부분에서 문제가 발생한다. 밑의 링크를 참조하여 수정하기 바란다. 기존 FLASK를 이용자들이 어느 정도 참고할만한 정보를 줬다. 굳이 각자 디버깅할 필요는 없고 맨 밑에 정상적으로 작동시키는데 성공한 내 코드를 적어둘테니 복사붙여넣기하면 된다. (단 OpenAI API를 결제하지 않으면 어차피 gpt-4o-mini에 대한 접근 권한이 없어서 작동하지는 않을 것이다)

https://github.com/kaistAI/FLASK/issues/4

또한 gpt-4-0613 모델이 더 이상 존재하지 않는다는 오류가 발생하는데 이걸 해결하는 방법은 다른 모델을 사용하는 방법 밖에 없고, 다른 모델들은 OpenAI API 홈페이지에서 결제를 하는 것 외에는 해결책이 없는 듯 하다. 밑은 gpt-4-0613 모델로 시도했을 경우 발생하는 오류 코드의 내용의 일부이다. openai migrate나 0.28 버전의 openai 패키지를 설치해봤지만 효과는 없었다.


'error': "APIRemovedInV1('\\n\\nYou tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.\\n\\nYou can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. \\n\\nAlternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`\\n\\nA detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742\\n')"}
@@@ []

OpenAI API를 결제한 후에 이용 가능한 모델을 선택하면 정상적으로 진행할 수 있다.

# 연합 모델 평가

In [ ]:
#OpenAI API를 호출하는 코드
evaluation_code = "/content/drive/MyDrive/OpenFedFLASK/FLASK/gpt_review/gpt4_eval.py"

#====================사용자의 입력 필요====================
#연합 학습된 모델의 이름
model_1 = "Mistral_FlaskDataset_c5s3"
model_2 = "Prometheus_FlaskDataset_c5s3"
model_name = model_1

#연합 학습 라운드 횟수
iteration = 10

#질문의 수
question_num = 1000
#==========================================================

#평가용 질문지 파일
question_file = f"/content/drive/MyDrive/OpenFedFLASK/FLASK/evaluation_set/flask_evaluation_q{question_num}.jsonl"

#모델의 추론 결과물 파일 경로
answer_file = f"/content/drive/MyDrive/OpenFedFLASK/flask_inference_outputs/{model_name}/q{question_num}_i{iteration}.jsonl"

#gpt-4의 리뷰 결과물 파일 경로
output_directory = f"/content/drive/MyDrive/OpenFedFLASK/flask_evaluation_outputs/{model_name}/q{question_num}_i{iteration}"
output_review_file = f"/content/drive/MyDrive/OpenFedFLASK/flask_evaluation_outputs/{model_name}/q{question_num}_i{iteration}/review.jsonl"

#리뷰 중 오류 저장 파일 경로
output_error_file = f"/content/drive/MyDrive/OpenFedFLASK/flask_evaluation_outputs/{model_name}/q{question_num}_i{iteration}/error.jsonl"

In [ ]:
#경로가 잘못되었거나 파일이 이미 존재한다면 Error 발생
create_empty_directory(output_directory)
create_empty_jsonl_file(output_review_file)
create_empty_jsonl_file(output_error_file)

Error: file '/content/drive/MyDrive/OpenFedFLASK/flask_evaluation_outputs/Mistral_FlaskDataset_c5s3/q1000_i10' exists.
Error: file '/content/drive/MyDrive/OpenFedFLASK/flask_evaluation_outputs/Mistral_FlaskDataset_c5s3/q1000_i10/review.jsonl' exists.
Error: file '/content/drive/MyDrive/OpenFedFLASK/flask_evaluation_outputs/Mistral_FlaskDataset_c5s3/q1000_i10/error.jsonl' exists.


In [ ]:
!python $evaluation_code --question-file $question_file --answer-file $answer_file --output-review-file $output_review_file --output-error-file $output_error_file --max-tokens 1024

Traceback (most recent call last):
  File "/content/drive/MyDrive/OpenFedFLASK/FLASK/gpt_review/gpt4_eval.py", line 97, in <module>
    key_jsons = get_json_list(args.key_file)
  File "/content/drive/MyDrive/OpenFedFLASK/FLASK/gpt_review/gpt4_eval.py", line 81, in get_json_list
    return json.load(f)
  File "/usr/lib/python3.10/json/__init__.py", line 293, in load
    return loads(fp.read(),
  File "/usr/lib/python3.10/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
  File "/usr/lib/python3.10/json/decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
  File "/usr/lib/python3.10/json/decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 3 column 6 (char 26)


# 일반 모델 평가

In [ ]:
#OpenAI API를 호출하는 코드
evaluation_code = "/content/drive/MyDrive/OpenFedFLASK/FLASK/gpt_review/gpt4_eval.py"

#====================사용자의 입력 필요====================
#연합 학습된 모델의 이름
model_1 = "Mistral_FlaskDataset"
model_2 = "Prometheus_FlaskDataset"
model_name = model_1

#연합 학습 라운드 횟수
iteration = 10

#질문의 수
question_num = 1000
#==========================================================

#평가용 질문지 파일
question_file = f"/content/drive/MyDrive/OpenFedFLASK/FLASK/evaluation_set/flask_evaluation_q{question_num}.jsonl"

#모델의 추론 결과물 파일 경로
answer_file = f"/content/drive/MyDrive/OpenFedFLASK/flask_inference_outputs/{model_name}/q{question_num}_i{iteration}.jsonl"

#gpt-4의 리뷰 결과물 파일 경로
output_directory = f"/content/drive/MyDrive/OpenFedFLASK/flask_evaluation_outputs/{model_name}/q{question_num}_i{iteration}"
output_review_file = f"/content/drive/MyDrive/OpenFedFLASK/flask_evaluation_outputs/{model_name}/q{question_num}_i{iteration}/review.jsonl"

#리뷰 중 오류 저장 파일 경로
output_error_file = f"/content/drive/MyDrive/OpenFedFLASK/flask_evaluation_outputs/{model_name}/q{question_num}_i{iteration}/error.jsonl"

In [ ]:
#경로가 잘못되었거나 파일이 이미 존재한다면 Error 발생
create_empty_directory(output_directory)
create_empty_jsonl_file(output_review_file)
create_empty_jsonl_file(output_error_file)

Error: file '/content/drive/MyDrive/OpenFedFLASK/flask_evaluation_outputs/Mistral_FlaskDataset_c5s3/q1000_i10' exists.
Error: file '/content/drive/MyDrive/OpenFedFLASK/flask_evaluation_outputs/Mistral_FlaskDataset_c5s3/q1000_i10/review.jsonl' exists.
Error: file '/content/drive/MyDrive/OpenFedFLASK/flask_evaluation_outputs/Mistral_FlaskDataset_c5s3/q1000_i10/error.jsonl' exists.


In [ ]:
!python $evaluation_code --question-file $question_file --answer-file $answer_file --output-review-file $output_review_file --output-error-file $output_error_file --max-tokens 1024

Traceback (most recent call last):
  File "/content/drive/MyDrive/OpenFedFLASK/FLASK/gpt_review/gpt4_eval.py", line 97, in <module>
    key_jsons = get_json_list(args.key_file)
  File "/content/drive/MyDrive/OpenFedFLASK/FLASK/gpt_review/gpt4_eval.py", line 81, in get_json_list
    return json.load(f)
  File "/usr/lib/python3.10/json/__init__.py", line 293, in load
    return loads(fp.read(),
  File "/usr/lib/python3.10/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
  File "/usr/lib/python3.10/json/decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
  File "/usr/lib/python3.10/json/decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 3 column 6 (char 26)
